In [12]:
from pathlib import Path
import json
from datetime import datetime

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [13]:
SILVER_FOLDER = Path("../../Data/silver")
SILVER_NLP_FOLDER = Path("../../Data/silver_nlp")
GOLD_FOLDER = Path("../../Data/gold")

GOLD_FOLDER.mkdir(parents=True, exist_ok=True)

In [14]:
# pip install transformers torch sentencepiece

In [15]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [16]:
def summarize_text(text, max_new_tokens=160):
    if not text.strip():
        return ""

    prompt = f"Vat de volgende tekst duidelijk samen in het Nederlands:\n\n{text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        do_sample=False
    )

    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary.strip()

In [17]:
def summarize_chunks(chunks, max_chunks=12):
    chunk_summaries = []

    for chunk in chunks[:max_chunks]:
        text = chunk["text"]

        if len(text.split()) < 40:
            continue

        summary = summarize_text(text, max_new_tokens=120)

        chunk_summaries.append({
            "chunk_id": chunk["chunk_id"],
            "summary": summary
        })

    return chunk_summaries

In [18]:
def create_final_summary(chunk_summaries):
    combined_text = " ".join(item["summary"] for item in chunk_summaries)

    if len(combined_text.split()) < 40:
        return combined_text

    return summarize_text(combined_text, max_new_tokens=220)


In [19]:
def get_modeling_hints(nlp_data):
    return {
        "main_topics": nlp_data.get("modeling_hints", {}).get("main_topics", []),
        "important_entities": nlp_data.get("modeling_hints", {}).get("important_entities", [])
    }

In [20]:
def save_gold_output(document_id, final_summary, chunk_summaries, modeling_hints):
    output_path = GOLD_FOLDER / f"{document_id}_gold.json"

    gold_data = {
        "document_id": document_id,
        "summary": final_summary,
        "chunk_summaries": chunk_summaries,
        "modeling_hints_used": modeling_hints,
        "model": {
            "task": "local_summarization",
            "name": MODEL_NAME,
            "type": "local_open_source"
        },
        "processed_at": datetime.now().isoformat()
    }

    output_path.write_text(
        json.dumps(gold_data, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"Saved Gold output: {output_path}")

In [21]:
def run_gold_layer():
    silver_files = sorted(SILVER_FOLDER.glob("doc_*_silver.json"))

    if not silver_files:
        print("No silver files found.")

    for silver_file in silver_files:
        document_id = silver_file.stem.replace("_silver", "")

        print(f"Generating summary for {document_id}...")

        silver_data = json.loads(silver_file.read_text(encoding="utf-8"))

        nlp_file = SILVER_NLP_FOLDER / f"{document_id}_nlp.json"

        if nlp_file.exists():
            nlp_data = json.loads(nlp_file.read_text(encoding="utf-8"))
            modeling_hints = get_modeling_hints(nlp_data)
        else:
            modeling_hints = {
                "main_topics": [],
                "important_entities": []
            }

        chunks = silver_data["chunks"]

        chunk_summaries = summarize_chunks(chunks)
        final_summary = create_final_summary(chunk_summaries)

        save_gold_output(
            document_id=document_id,
            final_summary=final_summary,
            chunk_summaries=chunk_summaries,
            modeling_hints=modeling_hints
        )

    print("Gold layer completed.")

In [22]:
run_gold_layer()

Generating summary for doc_01...
Saved Gold output: ..\..\Data\gold\doc_01_gold.json
Gold layer completed.
